# 05 · Conectando via Spark Connect (Caso B)

**Teoria**: docs/05-pyspark-na-pratica.md

**Pré-requisito**: `make up-cluster` (Spark Standalone: 1 master + 2 workers + um servidor Spark Connect, tudo em Docker).

🎯 **Objetivo**: conectar-se a um cluster Spark remoto usando o protocolo gRPC do Spark Connect. Seu processo Python aqui é um **cliente gRPC leve** — sem JVM, sem classpath do Hadoop, nada pesado.

Todo o processamento pesado acontece dentro do container `spark-connect`. Isso também significa: **os caminhos de arquivos que você referencia devem existir dentro dos containers**, não no seu laptop — é exatamente por isso que o `docker-compose.yml` faz bind-mount de `./data` em `/data` em cada serviço Spark.

In [ ]:
import sys

sys.path.insert(0, "../scripts")
from lab_utils import layer_path
from pyspark.sql import SparkSession

# Casos B/D: cliente leve sobre Spark Connect
# make up-cluster iniciou o servidor Spark Connect no container
# Todo o processamento pesado acontece no cluster Docker — este processo apenas envia
# o plano lógico não resolvido via gRPC e recebe os resultados de volta.
# Lembrete: qualquer caminho de arquivo (ex.: "/data/bronze/vendas") é resolvido
# DENTRO dos containers, não no seu laptop.
spark = (
    SparkSession.builder.appName("05-spark-connect")
    .remote("sc://localhost:15002")   # Endpoint gRPC do servidor Spark Connect
    .getOrCreate()
)
spark

## Comprovando que a execução realmente acontece no cluster

🎯 **Objetivo**: verificar que o processamento está rodando no cluster Docker, não localmente.

`spark.range(...).count()` é barato o suficiente para rodar em qualquer lugar — a verificação interessante é confirmar que a **Spark UI** que você vê é a do cluster.

Abra http://localhost:4040 (a UI do driver do servidor Connect) e http://localhost:8080 (a UI do Master Standalone) no seu navegador enquanto executa a próxima célula e veja o Job aparecer em tempo real.

In [ ]:
# Cria um DataFrame distribuído de 0 a 20 milhões — a execução acontece no cluster
# O cliente envia o plano lógico via gRPC, o servidor Spark Connect compila e executa
df = spark.range(0, 20_000_000)
print(f"Row count: {df.count():,}")
print("Check http://localhost:8080 -> Running Applications, and")
print("      http://localhost:4040 -> Jobs tab, to see this execution.")

📌 **Verificando na Spark UI**:

No http://localhost:8080 (Master), você deve ver uma aplicação chamada `05-spark-connect` na lista **Running Applications**. No http://localhost:4040, a aba **Jobs** mostra um Job com 1 Stage contendo 200 Tasks (partições padrão para 20M linhas).

💡 **Dica**: se não aparecer, pode ser que o cluster não esteja no ar — execute `make up-cluster` no terminal e espere os containers ficarem prontos.

## Lendo o dataset compartilhado

📌 Lembrete: `/data/...` é o caminho **dentro dos containers** (veja o mount de volume `x-spark-common` do `docker-compose.yml`).

Execute `make generate-data SCALE=large` no host primeiro — como `./data` está montado por bind, os containers o veem imediatamente, sem necessidade de etapa de cópia.

⚠️ **Atenção**: se o diretório não existir dentro do container, a leitura Parquet falhará com erro `Path does not exist`.

In [ ]:
# Lê os dados Parquet — os arquivos estão no bind-mount /data dentro do container
# layer_path("connect", ...) gera o caminho /data/bronze/vendas (visível dentro do container)
vendas = spark.read.parquet(layer_path("connect", "bronze", "vendas"))
categorias = spark.read.parquet(layer_path("connect", "bronze", "categorias"))

# O count() força a leitura distribuída — cada executor lê uma parte dos arquivos Parquet
print(f"vendas: {vendas.count():,} rows")
categorias.show(5)

📌 **Observação**:

O Spark Connect retornou os dados normalmente, como se fosse uma SparkSession local. A diferença? Toda a computação (leitura, descompressão, contagem) ocorreu nos workers do cluster — seu processo Python apenas recebeu os resultados.

💡 **Dica**: abra a aba **Stages** da Spark UI para ver as Tasks distribuídas entre os 2 workers do cluster.

## Tour pelas Partitions

🧠 **Por quê?** — o cliente Python do Spark Connect intencionalmente **não** expõe a API RDD de baixo nível (`df.rdd`) — é um cliente apenas de DataFrame/SQL. Para inspecionar as partitions, use `spark_partition_id()` como uma coluna comum.

📌 Cada linha pertencerá a uma partition diferente. A contagem por `partition_id` revela como os dados estão distribuídos entre os workers.

In [ ]:
from pyspark.sql.functions import spark_partition_id

# Agrupa por partition_id para ver quantas linhas cada partição contém
# spark_partition_id() é uma função que retorna o ID da partição de cada linha
partition_counts = (
    vendas.groupBy(spark_partition_id().alias("partition_id"))
    .count()
    .orderBy("partition_id")   # Ordena para facilitar a leitura
)
partition_counts.show(20)
print(f"Total partitions: {partition_counts.count()}")

📌 **Análise da distribuição**:

Observe se as partições têm tamanhos equilibrados (ideal) ou se há skew (algumas partições com muito mais linhas que outras). Dados desbalanceados podem causar Tasks lentas que atrasam o Job inteiro (problema de *data skew*).

💡 **Dica**: se houver skew, considere reparticionar com `.repartition(n, col)` ou habilitar AQE (Adaptive Query Execution) para balanceamento automático.

In [ ]:
# Encerra a SparkSession — a conexão gRPC com o servidor Spark Connect é fechada
# O cluster Docker continua rodando (make up-cluster ainda ativo)
spark.stop()